# Exploratory Data Analysis (EDA) — Titanic Dataset

Before doing any cleaning, feature engineering, or modeling, it's good practice to ask a standard set of questions about a new dataset. This notebook works through that checklist on the classic Titanic `train.csv` dataset.

## Setup

Load `pandas` and read the CSV into a DataFrame.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('train.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

# 1. How big is the data?

`df.shape` returns `(rows, columns)` — the fastest way to get a sense of the dataset's size before doing anything else. 891 passengers, 12 columns.

In [ ]:
df.shape

# 2. How does the data look like?

`df.sample(5)` shows 5 **random** rows — often more revealing than `df.head()`, since the first few rows of a file can be unrepresentative (e.g. sorted, or all from the same category).

In [ ]:
df.sample(5)

# 3. What is the data type of cols?

`df.info()` is the single most useful first command on any new dataset — it shows every column's dtype, how many non-null values it has, and total memory usage, all at once. Already visible here: `Age`, `Cabin`, and `Embarked` have fewer than 891 non-null values, meaning they contain missing data — confirmed properly in the next step.

In [ ]:
df.info()

# 4. Are there any missing values?

`df.isnull().sum()` counts missing values **per column** — the precise version of what `.info()` hinted at. `Cabin` is missing 687 of 891 values (77%), `Age` is missing 177 (20%), and `Embarked` only 2.

In [ ]:
df.isnull().sum()

# 5. How does the data look mathematically?

`df.describe()` gives summary statistics (count, mean, std, min, quartiles, max) for every **numeric** column. Useful for spotting outliers or implausible values — e.g. `Fare`'s max of 512 versus a median of ~14 hints at a right-skewed distribution with some very expensive tickets.

In [ ]:
df.describe()

### Separating Categorical from Numerical Columns — `select_dtypes()`

`describe()` above only summarized numeric columns by default. `select_dtypes()` lets you explicitly split the DataFrame by dtype — useful for deciding which columns need `.value_counts()`-style exploration (categorical) versus `.describe()`-style summary statistics (numerical).

In [ ]:
numerical_cols = df.select_dtypes(include='number').columns
categorical_cols = df.select_dtypes(include='object').columns

print('Numerical:', list(numerical_cols))
print('Categorical:', list(categorical_cols))


# 6. Are there duplicate values?

`df.duplicated().sum()` counts fully duplicate rows (every column identical). Zero here — as expected, since `PassengerId` alone is already a unique identifier.

In [ ]:
df.duplicated().sum()

# 7. How is the correlation between cols?

`df.corr()` computes the pairwise **Pearson correlation** between numeric columns — how strongly each moves in sync with `Survived`. Here, `Pclass` (-0.34) and `Fare` (0.26) show the strongest relationships: better ticket class and higher fare are both associated with a higher chance of survival.

### ⚠️ Important: `.corr()` Needs `numeric_only=True` on Modern Pandas

This DataFrame has non-numeric columns (`Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`). Older pandas silently ignored them when computing `.corr()`. Since **pandas 2.0**, the default changed — calling `.corr()` on a DataFrame with non-numeric columns now raises `ValueError: could not convert string to float` unless you explicitly pass `numeric_only=True`. The cell below is the fixed, version-safe equivalent of the original.

In [ ]:
df.corr(numeric_only=True)['Survived']


## 8. How Are the Categorical Columns Distributed?

`describe()` and `corr()` above only look at numeric columns. For categorical columns, `.value_counts()` is the equivalent starting point — it shows every unique value and how often it occurs, which is the fastest way to spot class imbalance (e.g. far more 3rd-class passengers than 1st-class).

In [ ]:
print(df['Survived'].value_counts())
print()
print(df['Pclass'].value_counts())
print()
print(df['Sex'].value_counts())
print()
print(df['Embarked'].value_counts())


## 9. How Many Unique Values Does Each Column Have?

`df.nunique()` counts distinct values per column in one line — a quick way to tell categorical columns (few unique values, like `Sex` with 2) apart from ones that are effectively unique identifiers (`PassengerId`, `Ticket`, `Name`, all close to 891 — not useful for grouping or modeling as categories).

In [ ]:
df.nunique().sort_values()


## 10. How Does Survival Compare Across Groups? — `groupby()`

`.corr()` only captures *linear* numeric relationships, and says nothing about categorical columns like `Sex` or `Pclass`. `groupby()` fills that gap directly: group the data by a category, then look at the survival rate within each group. This is usually the single most informative EDA step on this dataset — it's how the well-known "women and children first, and 1st class fared better" pattern actually shows up in the numbers.

In [ ]:
print(df.groupby('Sex')['Survived'].mean())
print()
print(df.groupby('Pclass')['Survived'].mean())


### Cross-Tabulating Two Categorical Columns — `pd.crosstab()`

`pd.crosstab()` is the categorical-vs-categorical equivalent of `.corr()` — it builds a table of counts (or, with `normalize=True`, proportions) for every combination of two columns' values.

In [ ]:
pd.crosstab(df['Pclass'], df['Survived'], normalize='index')


## Summary

| Question | Method |
|---|---|
| How big is the data? | `df.shape` |
| What does it look like? | `df.sample(n)` |
| What are the column types? | `df.info()` |
| Any missing values? | `df.isnull().sum()` |
| Numeric summary stats? | `df.describe()` |
| Split by column type | `df.select_dtypes(include=...)` |
| Any duplicate rows? | `df.duplicated().sum()` |
| Numeric correlations? | `df.corr(numeric_only=True)` |
| Categorical distributions? | `df['col'].value_counts()` |
| How many unique values? | `df.nunique()` |
| Compare a metric across groups? | `df.groupby('col')['target'].mean()` |
| Compare two categorical columns? | `pd.crosstab(col1, col2)` |

In [ ]:
df.corr()['Survived']